In [79]:
import string

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rafatmeraz/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rafatmeraz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rafatmeraz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Loading Data

In [80]:
df = pd.read_csv('./training_data.csv')

In [81]:
df.shape

(120000, 2)

In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    120000 non-null  str  
 1   label   120000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 28.9 MB


In [83]:
df.head(5)

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [84]:
df['label'].value_counts()

label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64

## Pre-processing

In [85]:
df.duplicated().sum()

np.int64(0)

In [86]:
ss = SnowballStemmer('english')

In [87]:
# Process the into lowercase
# Remove the stop words, special characters, emojis
# Do the word steaming
def transform_text(text):
    text = text.lower()

    text = nltk.word_tokenize(text)

    # Removing Special Characters
    y = []
    for i in text:
        if i.isalnum():
            y.append(i)

    # Removing stop words and punctuations
    text = y[:]
    y.clear()

    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)


    # Stemming using Snowball Stemmer
    text = y[:]
    y.clear()

    for i in text:
        y.append(ss.stem(i))

    return " ".join(y)

In [88]:
# Testing transform text function
transform_text("I am using Safari Browser and it's awesome")

'use safari browser awesom'

In [89]:
# Reduced df due to low config laptop
df = df[:1000]
df['transformed_text'] = df['text'].apply(transform_text)

df.head()

,text,label,transformed_text
0,Wall St. Bears Claw Back Into the Black (Reute...,2,wall bear claw back black reuter reuter wall s...
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2,carlyl look toward commerci aerospac reuter re...
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2,oil economi cloud stock outlook reuter reuter ...
3,Iraq Halts Oil Exports from Main Southern Pipe...,2,iraq halt oil export main southern pipelin reu...
4,"Oil prices soar to all-time record, posing new...",2,oil price soar record pose new menac us econom...


## Feature Engineering

In [90]:
tfidf = TfidfVectorizer(max_features=300)

X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['label'].values

In [91]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model training

In [95]:
models = {
    "lr": LogisticRegression(random_state=42, max_iter=100),
    "rfc": RandomForestClassifier(max_depth=5, min_samples_split=5),
    "abc": AdaBoostClassifier(n_estimators=100, learning_rate=0.1),
    "bc": BaggingClassifier(n_estimators=50, random_state=2),
    "xgbc": XGBClassifier(),
    "gbc": GradientBoostingClassifier(),
    "dtc": DecisionTreeClassifier(),
    "svc": SVC()
}

In [96]:
def train_classifier(clf, X_train, y_train, X_test, y_test):
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')

    return accuracy, precision

In [97]:
accuracy_scores = []
precision_scores = []

for name, model in models.items():
    accuracy, precision = train_classifier(model, X_train, y_train, X_test, y_test)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)

    print(f"For Model: {name}")
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")

For Model: lr
Accuracy: 0.76
Precision: 0.7768686868686868
For Model: rfc
Accuracy: 0.625
Precision: 0.779732343679712


/Users/rafatmeraz/PycharmProjects/news-classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


For Model: abc
Accuracy: 0.58
Precision: 0.5265445402298851
For Model: bc
Accuracy: 0.7
Precision: 0.6905190905190906
For Model: xgbc
Accuracy: 0.78
Precision: 0.7843951037772967
For Model: gbc
Accuracy: 0.755
Precision: 0.7707048611111111
For Model: dtc
Accuracy: 0.595
Precision: 0.606843218682822
For Model: svc
Accuracy: 0.76
Precision: 0.7855385247522887
